In [8]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("data/raw/titanic.csv")

In [2]:
y = df["Survived"]

X = df.drop("Survived", axis=1)

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

In [9]:
class FamilySizeTransformer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()

        X["FamilySize"] = (
            X["SibSp"] +
            X["Parch"] +
            1
        )

        return X

In [5]:
class TitleExtractor(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()

        X["Title"] = (
            X["Name"]
            .str.extract(r',\s*([^\.]+)\.')
        )

        return X

تست

In [6]:
t = TitleExtractor()

X_new = t.fit_transform(X)

## FunctionTransformer

وقتی کلاس لازم نیست

In [10]:
from sklearn.preprocessing import FunctionTransformer

In [11]:
def log_transform(x):
    return np.log1p(x)

ترنسفورمر:

In [12]:
log_transformer = FunctionTransformer(log_transform)

- Numeric Features
- categorical_features

In [13]:
numeric_features = ["Age","Fare","FamilySize"]
categorical_features = ["Sex","Embarked","Pclass","Title"]

## Numeric Pipeline

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [19]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median") ),
    ("scaler", StandardScaler())
])

## Categorical Pipeline

In [20]:
from sklearn.preprocessing import OneHotEncoder

In [21]:
categorical_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])

## ColumnTransformer

In [22]:
from sklearn.compose import ColumnTransformer

In [23]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline,     numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

## Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model = RandomForestClassifier(random_state=42)

## Full Pipeline

نکته مهم:

قبل از Preprocessing
فیچر انجینیرینگ انجام می‌شود.

In [ ]:
full_pipeline = Pipeline([
    ("family", FamilySizeTransformer()),
    ("title",TitleExtractor()),
    ("preprocessor", preprocessor),
    ("model",model)
])

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )
)

## Train

In [ ]:
full_pipeline.fit(
    X_train,
    y_train
)

## Predict

In [ ]:
pred = full_pipeline.predict(X_test)

## Grid Search

حالا قسمت حرفه‌ای.

خیلی‌ها اشتباه می‌کنند و فقط مدل را Tune می‌کنند.

ما کل Pipeline را Tune می‌کنیم.

### Parameter

In [ ]:
param_grid = {
    "model__n_estimators": [
        100,
        200,
        300
    ],
    "model__max_depth": [
        5,
        10,
        15
    ],
    "model__min_samples_split": [
        2,
        5,
        10
    ]
}